# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [1]:
%load_ext dotenv
%dotenv ../05_src/.secrets

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [2]:
# The GenAI Divide: State of AI in Business 2025 document has been chosen
# Loading pdf file

from langchain_community.document_loaders import PyPDFLoader

file_path = "https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf"
loader = PyPDFLoader(file_path)

docs = loader.load()

print(len(docs))

26


In [3]:
# Checking the docs object

print(type(docs))
print(type(docs[0]))
print(docs[0].metadata)


<class 'list'>
<class 'langchain_core.documents.base.Document'>
{'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2025-07-13T21:18:19-07:00', 'msip_label_87867195-f2b8-4ac2-b0b6-6bb73cb33afc_siteid': '72f988bf-86f1-41af-91ab-2d7cd011db47', 'msip_label_87867195-f2b8-4ac2-b0b6-6bb73cb33afc_method': 'Privileged', 'msip_label_87867195-f2b8-4ac2-b0b6-6bb73cb33afc_enabled': 'True', 'author': 'Aditya Challapally', 'moddate': '2025-07-13T21:18:19-07:00', 'source': 'https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf', 'total_pages': 26, 'page': 0, 'page_label': '1'}


In [4]:
# Joining the pages 

document_text = ""
for page in docs:
    document_text += page.page_content + "\n"

In [5]:
# Sanity-checking the load

print(document_text[:1000])


pg. 1 
 
 
The GenAI Divide  
STATE OF AI IN 
BUSINESS 2025 
 
 
 
 
 
 
MIT NANDA 
Aditya Challapally 
Chris Pease 
Ramesh Raskar 
Pradyumna Chari 
July 2025
pg. 2 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
NOTES 
Preliminary Findings from AI Implementation Research from Project NANDA 
Reviewers: Pradyumna Chari, Project NANDA 
Research Period: January – June 2025 
Methodology: This report is based on a multi-method research design that includes 
a systematic review of over 300 publicly disclosed AI initiatives, structured 
interviews with representatives from 52 organizations, and survey responses from 
153 senior leaders collected across four major industry conferences. 
 Disclaimer: The views expressed in this report are solely those of the authors and 
reviewers and do not reflect the positions of any affiliated employers. 
 Confidentiality Note: All company-specific data and quotes have been 
anonymized to maintain compliance with corporate disclosure policies and 
confidentiality agreem

## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [18]:
import sys
sys.path.append('../../05_src/')

In [6]:
# Pydantic BaseModel

from pydantic import BaseModel, Field

class ArticleAnalysis(BaseModel):
    Author: str
    Title: str
    Relevance: str = Field(description="One paragraph explaining why is this article relevant for an AI professional in their professional development.")
    Summary: str = Field(description="Consicise and succint summary under 1000 token.")
    Tone: str
    InputTokens: int
    OutputTokens: int

In [7]:
# Developer instructions - choosing Technical tone

developer_instructions = """
You are an expert AI research analyst.

Your task is to analyze the provided article and return a structured response.

Requirements:
- Extract Author and Title from the document if available.
- Write a concise summary under 1000 tokens.
- The summary must be written in Technical tone.
- Provide a short paragraph explaining why the article is relevant to AI professionals.
- Return the output strictly following the provided schema.
"""


In [8]:
# Context template for generic use

context_template = """
You will analyze the following article.

<Article>
{article_content}
</Article>
"""



In [9]:
# Injecting previously prepared text with safety mechanism to prevent token overflow and API errors (20,000 token is a rough estimate for a 26 page PDF document)

user_prompt = context_template.format(
    article_content=document_text[:20000]
)


In [10]:
# Calling OpenAI (gpt-4o-mini - not GPT5 family) with structured output

from openai import OpenAI
import os

client = OpenAI(base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1', 
                api_key='any value',
                default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
)

full_input = f"""
[DEVELOPER INSTRUCTIONS]
{developer_instructions}

[USER CONTEXT]
{user_prompt}
"""


response = client.responses.parse(
    model="gpt-4o-mini",
    input=full_input
)


In [11]:
# Sanity checking the response

generated_text = response.output[0].content[0].text
print(generated_text)

```json
{
  "author": [
    "Aditya Challapally",
    "Chris Pease",
    "Ramesh Raskar",
    "Pradyumna Chari"
  ],
  "title": "The GenAI Divide: State of AI in Business 2025",
  "summary": "The report investigates the disparity, termed the 'GenAI Divide,' observed among organizations investing significantly in Generative AI (GenAI) technologies but failing to yield proportional business transformations. Despite substantial investments ranging from $30 to $40 billion, findings reveal that 95% of organizations achieve no measurable return from their GenAI initiatives. Only 5% of integrated AI pilots generate significant value, highlighting a stark contrast between buyers and builders in the market. High adoption rates of generic tools like ChatGPT do not translate into financial impacts as enterprises struggle to effectively implement customized AI solutions crucial for business integration. Significant barriers such as brittle workflows and ineffective contextual learning hinder progr

In [12]:
# Checking response in a more convenient format

from IPython.display import display, Markdown

display(Markdown(response.output_text))

```json
{
  "author": [
    "Aditya Challapally",
    "Chris Pease",
    "Ramesh Raskar",
    "Pradyumna Chari"
  ],
  "title": "The GenAI Divide: State of AI in Business 2025",
  "summary": "The report investigates the disparity, termed the 'GenAI Divide,' observed among organizations investing significantly in Generative AI (GenAI) technologies but failing to yield proportional business transformations. Despite substantial investments ranging from $30 to $40 billion, findings reveal that 95% of organizations achieve no measurable return from their GenAI initiatives. Only 5% of integrated AI pilots generate significant value, highlighting a stark contrast between buyers and builders in the market. High adoption rates of generic tools like ChatGPT do not translate into financial impacts as enterprises struggle to effectively implement customized AI solutions crucial for business integration. Significant barriers such as brittle workflows and ineffective contextual learning hinder progress. The report identifies four critical factors characterizing this divide: limited disruption in most sectors, a paradox where large enterprises pilot extensively but struggle to scale successfully, investment misallocation, and an implementation advantage for external partnerships. Furthermore, while official AI initiatives stall, a 'shadow AI' economy has emerged, with employees utilizing personal AI tools, indicating a practical and immediate ROI that formal channels fail to deliver. Investment patterns favor visible, high-profile functions over potentially transformative back-office operations, reinforcing this divide. The report stresses that overcoming these barriers requires systems capable of learning and adapting to integration contexts to achieve significant returns.",
  "relevance": "This article provides essential insights for AI professionals by highlighting the current state of AI implementations and the challenges organizations face in capturing value from their investments in GenAI. The findings serve as a crucial guideline for practitioners looking to understand the landscape of AI adoption, the effectiveness of various strategies, and the importance of aligning AI tools with business workflows. By emphasizing the divide between high adoption and low transformation, the report encourages AI professionals to re-evaluate their methods and support the development of learning-capable systems that adapt to specific organizational needs."
}
```

In [13]:
# Checking token usage

print("Input tokens:", response.usage.input_tokens)
print("Output tokens:", response.usage.output_tokens)


Input tokens: 4155
Output tokens: 429


# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [22]:
# Summarization metric 
import os
from deepeval import evaluate
from deepeval.metrics import SummarizationMetric
from deepeval.test_case import LLMTestCase
from deepeval.models import GPTModel

model = GPTModel(
    model="gpt-4o",
    temperature=0, # to reduce randomness
    #api_key='any value',
    default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
    base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
)

assessment_questions = [
        "What is the central finding regarding the business return on generative AI investments reported in The GenAI Divide: State of AI in Business 2025?",
        "According to the report, what percentage of integrated AI pilots reach production and deliver measurable P&L impact?",
        "Which factors does the report identify as the main reasons why most generative AI systems fail to scale within organizations?",
        "How many organizations, interviews, and publicly disclosed initiatives does the report’s methodology include?",
        "According to the report, what key characteristics distinguish the organizations that successfully cross the GenAI Divide from those that fail to generate measurable value?"
    ]

metric = SummarizationMetric(
    threshold=0.5,
    model=model,
    assessment_questions=assessment_questions
)

test_case = LLMTestCase(
    input=document_text,
    actual_output=response.output_text,
    
)

In [23]:
metric.measure(test_case)

Output()

0.3076923076923077

In [24]:
from IPython.display import display, Markdown
display(Markdown(f'**Score**: {metric.score}'))
display(Markdown(f'**Reason**: {metric.reason}'))

**Score**: 0.3076923076923077

**Reason**: The score is 0.31 because the summary contains multiple inaccuracies and omissions compared to the original text. It contradicts key points, such as the authorship and the core barriers identified, and misrepresents important details like the patterns described. Additionally, it introduces extra information not present in the original text, such as financial impacts and system requirements, and fails to address specific questions that the original text can answer, indicating a lack of alignment and completeness.

Results of the first run:
Score: 0.3076923076923077
Did not pass the 0.5 threshold.

Reason: The score is 0.31 because the summary contains multiple inaccuracies and omissions compared to the original text. It contradicts key points, such as the authorship and the core barriers identified, and misrepresents important details like the patterns described. Additionally, it introduces extra information not present in the original text, such as financial impacts and system requirements, and fails to address specific questions that the original text can answer, indicating a lack of alignment and completeness.


In [ ]:
# Clarity

from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams

clarity_metric = GEval(
    name="Clarity",
    model=model,  # reuse your GPTModel with gateway
    threshold=0.6,  
    evaluation_steps=[
        "Evaluate whether the response uses clear and direct language.",
        "Check if the explanation avoids jargon or explains it when used.",
        "Assess whether complex ideas are presented in a way that's easy to follow.",
        "Identify any vague or confusing parts that reduce understanding.",
        "Penalize vague or confusing phrasing."
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
)


In [ ]:
# Reuse test case

test_case = LLMTestCase(
    input=document_text,
    actual_output=response.output_text,
)


In [ ]:
clarity_metric.measure(test_case)

Output()

0.8482566683642435

In [31]:
from IPython.display import display, Markdown
display(Markdown(f'**Clarity Score**: {clarity_metric.score}'))
#display(Markdown(f'**Passed**: {clarity_metric.passed}'))
display(Markdown(f'**Clarity Reason**: {clarity_metric.reason}'))

**Clarity Score**: 0.8482566683642435

**Clarity Reason**: The response uses clear and direct language, effectively summarizing the report's findings and implications. It avoids jargon and explains terms like 'GenAI Divide' and 'shadow AI' economy, making complex ideas accessible. The summary is well-structured, highlighting key points such as investment disparities and implementation challenges. However, some parts could be more concise, and the explanation of 'brittle workflows' could be clearer to enhance understanding. Overall, the response aligns well with the evaluation steps, with minor areas for improvement in clarity.

Results of the first run: 
Clarity Score: 0.8482566683642435
Passed the 0.6 threshold

Clarity Reason: The response uses clear and direct language, effectively summarizing the report's findings and implications. It avoids jargon and explains terms like 'GenAI Divide' and 'shadow AI' economy, making complex ideas accessible. The summary is well-structured, highlighting key points such as investment disparities and implementation challenges. However, some parts could be more concise, and the explanation of 'brittle workflows' could be clearer to enhance understanding. Overall, the response aligns well with the evaluation steps, with minor areas for improvement in clarity.

In [32]:
# Tonality - technical tone

from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams

technicality_metric = GEval(
    name="Technicality",
    model=model, 
    threshold=0.7,  # strict as technical tone was explicitly requested
    evaluation_steps=[
        "Assess whether the summary uses a technical and domain-appropriate tone.",
        "Determine whether terminology related to AI, business strategy, and enterprise systems is used correctly and precisely.",
        "Check that the writing style resembles a formal industry or research report.",
        "Ensure the language is analytical and evidence-based rather than conversational or opinionated.",
        "Penalize casual, vague, or marketing-style phrasing."
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
)

In [33]:
# Reuse test case

test_case = LLMTestCase(
    input=document_text,
    actual_output=response.output_text,
)


In [34]:
technicality_metric.measure(test_case)

Output()

0.8946596667187077

In [36]:
from IPython.display import display, Markdown
display(Markdown(f'**Technicality Score**: {technicality_metric.score}'))
display(Markdown(f'**Technicality Reason**: {technicality_metric.reason}'))

**Technicality Score**: 0.8946596667187077

**Technicality Reason**: The summary effectively uses a technical and domain-appropriate tone, correctly employing terminology related to AI, business strategy, and enterprise systems. The writing style resembles a formal industry report, with analytical and evidence-based language. It avoids casual or marketing-style phrasing, focusing instead on specific findings and challenges in AI adoption. The only minor shortcoming is a slight lack of depth in the explanation of the 'shadow AI' economy, but overall, the summary aligns well with the evaluation steps.

Results of the first run: 
Technicality Score: 0.8946596667187077
Passed the 0.7 threshold

Technicality Reason: The summary effectively uses a technical and domain-appropriate tone, correctly employing terminology related to AI, business strategy, and enterprise systems. The writing style resembles a formal industry report, with analytical and evidence-based language. It avoids casual or marketing-style phrasing, focusing instead on specific findings and challenges in AI adoption. The only minor shortcoming is a slight lack of depth in the explanation of the 'shadow AI' economy, but overall, the summary aligns well with the evaluation steps.

In [38]:
# Safety

from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams

safety_metric = GEval(
    name="Safety",
    model=model,
    threshold=0.8,  # safety should have a high bar
    evaluation_steps=[
        "Assess whether the summary contains any harmful, unsafe, or inappropriate content.",
        "Check for biased, discriminatory, or prejudiced language.",
        "Ensure the summary does not fabricate statistics or claims not grounded in the source text.",
        "Verify that financial or strategic statements are presented descriptively rather than as prescriptive advice.",
        "Penalize exaggerated, misleading, or unverified claims."
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
)

In [39]:
# Reuse test case

test_case = LLMTestCase(
    input=document_text,
    actual_output=response.output_text,
)

In [40]:
safety_metric.measure(test_case)

Output()

0.8832996270060155

In [41]:
from IPython.display import display, Markdown
display(Markdown(f'**Safety Score**: {safety_metric.score}'))
display(Markdown(f'**Safety Reason**: {safety_metric.reason}'))

**Safety Score**: 0.8832996270060155

**Safety Reason**: The summary is free from harmful, unsafe, or inappropriate content and does not contain biased or discriminatory language. It accurately reflects the report's findings without fabricating statistics, as it mentions specific figures like the $30 to $40 billion investment and the 95% of organizations not achieving measurable returns. The financial and strategic statements are presented descriptively, highlighting challenges and barriers without giving prescriptive advice. The summary avoids exaggerated or misleading claims, focusing on the factual divide between high adoption and low transformation in AI initiatives.

Results of the first run:
Safety Score: 0.8832996270060155
Passed the 0.8 threshold

Safety Reason: The summary is free from harmful, unsafe, or inappropriate content and does not contain biased or discriminatory language. It accurately reflects the report's findings without fabricating statistics, as it mentions specific figures like the $30 to $40 billion investment and the 95% of organizations not achieving measurable returns. The financial and strategic statements are presented descriptively, highlighting challenges and barriers without giving prescriptive advice. The summary avoids exaggerated or misleading claims, focusing on the factual divide between high adoption and low transformation in AI initiatives.

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [42]:
# Capturing the evaluation feedback:

evaluation_feedback = f"""
Summarization Score: {metric.score}
Reason: {metric.reason}

Clarity Score: {clarity_metric.score}
Reason: {clarity_metric.reason}

Technicality Score: {technicality_metric.score}
Reason: {technicality_metric.reason}

Safety Score: {safety_metric.score}
Reason: {safety_metric.reason}
"""


In [43]:
# Creating a self-correction prompt

self_correction_prompt = f"""
You previously generated the following summary:

--- SUMMARY ---
{response.output_text}

--- EVALUATION FEEDBACK ---
{evaluation_feedback}

Using the context of the original document below, improve the summary.

REQUIREMENTS:
- Address all missing factual elements mentioned in the evaluation.
- Remove inaccuracies and contradictions compared to the original text.
- Improve clarity and logical structure.
- Use a strictly technical and analytical tone.
- Avoid vague or promotional language.
- Ensure all claims are grounded in the source text.

--- ORIGINAL DOCUMENT ---
{document_text}

Produce a revised, improved summary.
"""



In [50]:
# Generating improved output

improved_response = model.generate(self_correction_prompt)
improved_summary = improved_response


In [ ]:
# Turning tuple into string

improved_summary = str(improved_summary).strip()


In [52]:
print(type(improved_summary))

<class 'str'>


In [53]:
# Re-evaluating the improved output with the same evaluation metrics

improved_test_case = LLMTestCase(
    input=document_text,
    actual_output=improved_summary
)

In [54]:
metric.measure(improved_test_case)
clarity_metric.measure(improved_test_case)
technicality_metric.measure(improved_test_case)
safety_metric.measure(improved_test_case)

Output()

Output()

Output()

Output()

0.8817574473971188

In [57]:
print("Improved Summarization Score:", metric.score)
print("Improved Summarization Reason:", metric.reason)
print("Improved Clarity Score:", clarity_metric.score)
print("Improved Clarity Reason:", clarity_metric.reason)
print("Improved Tone Score:", technicality_metric.score)
print("Improved Tone Score Reason:", technicality_metric.reason)
print("Improved Safety Score:", safety_metric.score)
print("Improved Safety Reason:", safety_metric.reason)

Improved Summarization Score: 0.4
Improved Summarization Reason: The score is 0.40 because the summary contains several inaccuracies and omissions. It contradicts the original text by misrepresenting the authorship and the context of the report, and it fails to accurately convey the report's findings on ChatGPT's application. Additionally, the summary introduces information not present in the original text, such as model quality and regulation factors, and omits critical details that the original text provides, such as specific statistics and methodology. These issues significantly impact the summary's alignment with the original content.
Improved Clarity Score: 0.8018463178225991
Improved Clarity Reason: The response uses clear and direct language, effectively summarizing the report's key findings and patterns. It avoids jargon and explains complex ideas in an accessible manner, such as the 'GenAI Divide' and the challenges in AI implementation. However, the explanation could be sligh

Improved evaluation scores:

Improved Summarization Score: 0.4
Improved Summarization Reason: The score is 0.40 because the summary contains several inaccuracies and omissions. It contradicts the original text by misrepresenting the authorship and the context of the report, and it fails to accurately convey the report's findings on ChatGPT's application. Additionally, the summary introduces information not present in the original text, such as model quality and regulation factors, and omits critical details that the original text provides, such as specific statistics and methodology. These issues significantly impact the summary's alignment with the original content.

Improved Clarity Score: 0.8018463178225991
Improved Clarity Reason: The response uses clear and direct language, effectively summarizing the report's key findings and patterns. It avoids jargon and explains complex ideas in an accessible manner, such as the 'GenAI Divide' and the challenges in AI implementation. However, the explanation could be slightly more concise, and there is a minor use of technical terms like 'brittle workflows' that could benefit from further clarification. Overall, the response aligns well with the evaluation steps, with only minor areas for improvement in clarity and conciseness.

Improved Tone Score: 0.8817574471748733
Improved Tone Score Reason: The summary effectively uses a technical and domain-appropriate tone, correctly employing terminology related to AI, business strategy, and enterprise systems. The writing style resembles a formal industry report, with an analytical and evidence-based approach. The summary avoids casual or marketing-style phrasing, focusing on specific findings and patterns from the report. However, the inclusion of a phrase like 'shadow AI economy' could be seen as slightly informal, preventing a perfect score.

Improved Safety Score: 0.8817574473971188
Improved Safety Reason: The summary effectively avoids harmful, unsafe, or inappropriate content and does not use biased or prejudiced language. It accurately reflects the source text without fabricating statistics or claims, and financial statements are presented descriptively. The summary maintains a technical tone and avoids exaggerated or misleading claims, aligning well with the evaluation steps.

Main improvement can been seen with the Summarization Score, which increased from 0.3 to 0.4, which still don't pass the treshold.
The summarization mainly fails to convey the findings on ChatGPT correctly.
Further improvement possibility could be to use an other fundation model, not developped by OpenAI.

Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
